In [1]:
import numpy as np
import pandas as pd
import OpenEXR
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image, ImageDraw

In [2]:
ds_path = Path(r"C:\Users\sol77\Desktop\ML-For-CV-Robustness\datasets\datasetGTCombined.csv")
Path.is_file(ds_path)

True

In [3]:
ds = pd.read_csv(ds_path)
ds

,grviIn,meanRefR,meanRefG,illumR,illumG,blockIdx,sunRoll,sunPitch,sunYaw,camRoll,camPitch,camYaw,grviOut
0,0.163549,0.000324,0.000421,1130.973355,1020.232214,0.0,180.0,-63.812729,-56.074120,0.0,-19.258,-0.0,0.20
1,0.097110,0.000412,0.000486,1130.973355,1020.232214,1.0,180.0,-63.812729,-56.074120,0.0,-19.258,-0.0,0.20
2,0.074345,0.000448,0.000517,1130.973355,1020.232214,2.0,180.0,-63.812729,-56.074120,0.0,-19.258,-0.0,0.20
3,0.156574,0.000374,0.000481,1130.973355,1020.232214,3.0,180.0,-63.812729,-56.074120,0.0,-19.258,-0.0,0.20
4,0.116627,0.000393,0.000479,1130.973355,1020.232214,4.0,180.0,-63.812729,-56.074120,0.0,-19.258,-0.0,0.20
...,...,...,...,...,...,...,...,...,...,...,...,...,...
667235,0.261726,0.000240,0.000362,553.313000,483.020000,15.0,180.0,-42.301559,-85.835487,-0.0,-22.218,-180.0,0.26
667236,0.239247,0.000270,0.000396,553.313000,483.020000,16.0,180.0,-42.301559,-85.835487,-0.0,-22.218,-180.0,0.26
667237,0.283672,0.000269,0.000422,553.313000,483.020000,17.0,180.0,-42.301559,-85.835487,-0.0,-22.218,-180.0,0.26
667238,0.241322,0.000345,0.000508,553.313000,483.020000,18.0,180.0,-42.301559,-85.835487,-0.0,-22.218,-180.0,0.26


In [4]:
ds_ar = ds.to_numpy()

In [5]:
f = np.array([1.0, 0.0, 0.0])
r = np.array([0.0, 1.0, 0.0])

In [ ]:
angles = []

for row in ds_ar:
    c =  np.deg2rad(np.array([row[9], row[10], row[11]]))
    s =  np.deg2rad(np.array([row[6], row[7], row[8]]))

    # camera
    c_r_cos = np.cos(c[0])
    c_r_sin = np.sin(c[0])
    
    c_p_cos = np.cos(c[1])
    c_p_sin = np.sin(c[1])
    
    c_y_cos = np.cos(c[2])
    c_y_sin = np.sin(c[2])
    
    # sun
    s_r_cos = np.cos(s[0])
    s_r_sin = np.sin(s[0])
    
    s_p_cos = np.cos(s[1])
    s_p_sin = np.sin(s[1])
    
    s_y_cos = np.cos(s[2])
    s_y_sin = np.sin(s[2])

    #camera
    Rxc = np.array([[1.0, 0.0, 0.0], [0.0, c_r_cos, -1.0 * c_r_sin], [0.0, c_r_sin, c_r_cos]])
    Ryc = np.array([[c_p_cos, 0.0, c_p_sin], [0.0, 1.0, 0.0], [-1*c_p_sin, 0.0, c_p_cos]])
    Rzc = np.array([[c_y_cos, -1*c_y_sin, 0.0], [c_y_sin, c_y_cos, 0.0], [0.0, 0.0, 1]])

    Rc = np.matmul(Rzc, np.matmul(Ryc, Rxc))
    
    #sun
    Rxs = np.array([[1.0, 0.0, 0.0], [0.0, s_r_cos, -1.0 * s_r_sin], [0.0, s_r_sin, s_r_cos]])
    Rys = np.array([[s_p_cos, 0.0, s_p_sin], [0.0, 1.0, 0.0], [-1*s_p_sin, 0.0, s_p_cos]])
    Rzs = np.array([[s_y_cos, -1*s_y_sin, 0.0], [s_y_sin, s_y_cos, 0.0], [0.0, 0.0, 1]])

    Rs = np.matmul(Rzs, np.matmul(Rys, Rxs))

    fc = np.matmul(Rc, f)
    rc = np.matmul(Rc, r)
    fs = np.matmul(Rs, f)

    sf = np.dot(fs, fc)
    sr = np.dot(fs, rc)

    angle = np.arctan2(sr, sf)

    angles.append(angle)

In [ ]:
angles_df = pd.DataFrame({"azimuth": angles})
angles_df

In [ ]:
ds_nu = ds.drop(["sunRoll", "sunPitch", "sunYaw", "camRoll", "camPitch", "camYaw"], axis=1)
ds_nu

In [ ]:
out = ds["grviOut"]
ds_nu = ds_nu.drop(["grviOut"], axis=1)
ds_nu

In [ ]:
ds_nu["azimuth"] = angles_df["azimuth"].iloc[:len(out)]
ds_nu

In [ ]:
ds_nu["grviOut"] = out.iloc[:len(out)]
ds_nu

In [ ]:
ds_nu.to_csv(r"C:\Users\sol77\Desktop\comp-gt-azim.csv", index=False)